## Part 2, Random Forests and SVMs

In [1]:
# importing libraries

import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report


In [2]:
# Importing Dataframe and sense checking

h10k = pd.read_csv('hansard10000.csv')

print(h10k.shape)
print(h10k.isnull().sum())
h10k.head(3)

(10000, 8)
speech             0
party            411
constituency     411
date               0
speech_class       0
major_heading      0
year               0
speakername        0
dtype: int64


,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson
2,"The hon. Lady is wrong; as a matter of fact, w...",Conservative,Newark,2021-02-10,Speech,Building Safety,2021,Robert Jenrick


In [3]:
# Listing out the different parties, the different entries in 'speech_class'
# counting the nr. of speeches less thatn 1000 characters

print(h10k['party'].value_counts(), '\n')
print(h10k['speech_class'].value_counts(), '\n')
print("Nr. of speeches under 1000 ch.:", (h10k['speech'].str.len() < 1000).sum())

party
Conservative                        6192
Labour                              1828
Scottish National Party              560
Labour (Co-op)                       280
Liberal Democrat                     221
Speaker                              208
Democratic Unionist Party            140
Independent                           58
Plaid Cymru                           51
Social Democratic & Labour Party      21
Green Party                           17
Alliance                              12
Alba Party                             1
Name: count, dtype: int64 

speech_class
Speech        9614
Procedural     343
Division        43
Name: count, dtype: int64 

Nr. of speeches under 1000 ch.: 7752


### Post hand in edit 
- 01/07/2026
- Dropping Lib Dem Party as there are not enough examples of speeches to train models meaningsfully

In [6]:
# Replacing 'Labour Co-op' with 'Labour', dropping all parties except top 4, sense checking

h10k['party'] = h10k['party'].replace('Labour (Co-op)', 'Labour')
h10k = h10k[h10k['party'].isin(['Conservative', 'Labour', 'Scottish National Party'])]

print(h10k.isnull().sum())
h10k['party'].value_counts()

speech           0
party            0
constituency     0
date             0
speech_class     0
major_heading    0
year             0
speakername      0
dtype: int64


party
Conservative               1247
Labour                      626
Scottish National Party     166
Name: count, dtype: int64

In [7]:
# Removing ll non 'speech' entries from 'speech class'
# Removing all speeches under 1000 characters and sense checking

h10k = h10k[h10k['speech_class'].isin(['Speech'])]

h10k = h10k[h10k['speech'].str.len() > 1000]

print('Shape of the dataframe:', h10k.shape)

h10k.head()


Shape of the dataframe: (2039, 8)


,speech,party,constituency,date,speech_class,major_heading,year,speakername
2,"The hon. Lady is wrong; as a matter of fact, w...",Conservative,Newark,2021-02-10,Speech,Building Safety,2021,Robert Jenrick
5,"Given that this was outlined earlier today, it...",Conservative,Great Yarmouth,2021-04-13,Speech,Northern Ireland,2021,Brandon Lewis
7,The aftermath of the 2008 crisis saw not only ...,Labour,"Sheffield, Hallam",2021-02-23,Speech,Government's Management of the Economy,2021,Olivia Blake
17,The right hon. Gentleman puts it correctly. Wh...,Conservative,Sutton and Cheam,2020-12-07,Speech,United Kingdom Internal Market Bill,2020,Paul Scully
18,"As always, I am grateful to the hon. Gentleman...",Conservative,North East Somerset,2021-01-28,Speech,Business of the House,2021,Jacob Rees-Mogg


### Post hand in update
- In initial version (still available in 'Hand in project - NH/.py') there was an initial RF & SVM setup
- This then had uni/bi/trigrams added as parameters
- Then I experimented with building my own tokenizer

I have replaced this with a function that sets up the RF and SVMs with n-grams.
Adding the custom tokenizer is optional.

In [12]:
def rf_svm(x, y):
    '''
    This function takes two columns of input from a datafram and uses them to train a random forest model and
    an SVM. 

    The vectorizer uses a standard tokenizer, filters outs english stopwords, caps features at 3000 and takes in uni- and bigrams.

    Using only uni- and bigrams proved more effective than also adding trigrams.

    It then prints f1 scores and prediction report based on the classification.
    '''

    vec_h10k = TfidfVectorizer(stop_words = 'english', max_features = 3000, ngram_range = (1, 2))        # Setting parameters for vectors
    x_vec = vec_h10k.fit_transform(x)                                                                    # Vectorizing the text

    x_train, x_test, y_train, y_test = train_test_split(x_vec, y, stratify = y, random_state = 26)       # Splitting the set

    rf = RandomForestClassifier(n_estimators = 300, random_state = 26).fit(x_train, y_train)             # Random Forest
    support_vm = SVC(kernel = 'linear', random_state = 26).fit(x_train, y_train) 

    for name, clf in [('RandomForest', rf), ('SVM', support_vm)]:                                        # SVM
        party_pred = clf.predict(x_test)
        print(name)
        print('Macro avg. F1:', f1_score(y_test, party_pred, average = 'macro', zero_division = 0))
        print(classification_report(y_test, party_pred, zero_division = 0))
    

In [13]:
rf_svm(h10k['speech'], h10k['party'])

RandomForest
Macro avg. F1: 0.6127588752795138
                         precision    recall  f1-score   support

           Conservative       0.74      0.95      0.83       312
                 Labour       0.76      0.46      0.58       157
Scottish National Party       0.80      0.29      0.43        41

               accuracy                           0.75       510
              macro avg       0.77      0.57      0.61       510
           weighted avg       0.75      0.75      0.72       510

SVM
Macro avg. F1: 0.6622935279850174
                         precision    recall  f1-score   support

           Conservative       0.77      0.89      0.82       312
                 Labour       0.70      0.55      0.62       157
Scottish National Party       0.72      0.44      0.55        41

               accuracy                           0.75       510
              macro avg       0.73      0.63      0.66       510
           weighted avg       0.74      0.75      0.74       510


### Creating a custom tokenizer

In [16]:
# Importing necessary libraries

import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.corpus import stopwords

lemmatizer = WordNetLemmatizer()                  # Setting some variables for the function
stop_w = set(stopwords.words('english'))

In [17]:
def custom_toke(text):
    ''' 
    Takes words and tokenizes them 
    First puts all letters into lowercase, then drops anything that's not letters
    Lemmatizes for consistency across similar words
    Removes stopwords
    '''

    text = text.lower()
    toke = re.findall(r'[a-z]+', text)
    toke = [t for t in toke if len(t) > 2]
    toke = [lemmatizer.lemmatize(t, pos = 'v') for t in toke]     # Adding verbs into lemmatizer mix
    toke = [t for t in toke if t not in stop_w]
    
    return toke

In [25]:
def rf_svm_ct(x, y):
    '''
    Same function as previous but uses a custom tokenizer
    Max features have been dropped to 2000.
    '''

    vec_h10k = TfidfVectorizer(tokenizer = custom_toke, stop_words = 'english',\
                               max_features = 2000, ngram_range = (1, 2))                                # Setting parameters for vectors
    x_vec = vec_h10k.fit_transform(x)                                                                    # Vectorizing the text

    x_train, x_test, y_train, y_test = train_test_split(x_vec, y, stratify = y, random_state = 26)       # Splitting the set

    rf = RandomForestClassifier(n_estimators = 300, random_state = 26).fit(x_train, y_train)             # Random Forest
    support_vm = SVC(kernel = 'linear', random_state = 26).fit(x_train, y_train) 

    for name, clf in [('RandomForest', rf), ('SVM', support_vm)]:                                        # SVM
        party_pred = clf.predict(x_test)
        print(name)
        print('Macro avg. F1:', f1_score(y_test, party_pred, average = 'macro', zero_division = 0))
        print(classification_report(y_test, party_pred, zero_division = 0))

In [26]:
rf_svm_ct(h10k['speech'], h10k['party'])

C:\Users\nikki\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\nikki\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['make'] not in stop_words.
  warnings.warn(


RandomForest
Macro avg. F1: 0.5745914191963454
                         precision    recall  f1-score   support

           Conservative       0.73      0.96      0.83       312
                 Labour       0.79      0.46      0.58       157
Scottish National Party       0.80      0.20      0.31        41

               accuracy                           0.74       510
              macro avg       0.77      0.54      0.57       510
           weighted avg       0.76      0.74      0.71       510

SVM
Macro avg. F1: 0.6685285663200747
                         precision    recall  f1-score   support

           Conservative       0.79      0.89      0.84       312
                 Labour       0.69      0.59      0.64       157
Scottish National Party       0.74      0.41      0.53        41

               accuracy                           0.76       510
              macro avg       0.74      0.63      0.67       510
           weighted avg       0.75      0.76      0.75       510


In [14]:
# Testing out the new tokenizer

vec_h10k_custom = TfidfVectorizer(tokenizer = custom_toke, max_features = 2000)       # Setting parameters for vectors

x2 = vec_h10k_custom.fit_transform(h10k['speech'])
y2 = h10k['party']

x2_train, x2_test, y2_train, y2_test = train_test_split(x2, y2, stratify = y2, random_state = 26)       # Splitting the set

rando2 = RandomForestClassifier(n_estimators = 300, random_state = 26).fit(x2_train, y2_train)      # Random Forest
support_vm2 = SVC(kernel = 'linear', random_state = 26).fit(x2_train, y2_train)                     # Support Vector Machine

C:\Users\nikki\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [15]:
# Classification of parties and F1 score

for name2, clf in [('RandomForest', rando2), ('SVM', support_vm2)]:
    party_pred2 = clf.predict(x2_test)
    print(name2)
    print('Macro avg. F1:', f1_score(y2_test, party_pred2, average = 'macro', zero_division = 0))
    print(classification_report(y2_test, party_pred2, zero_division = 0))

RandomForest
Macro avg. F1: 0.41681757259865976
                         precision    recall  f1-score   support

           Conservative       0.69      0.97      0.81       312
                 Labour       0.75      0.38      0.51       157
       Liberal Democrat       0.00      0.00      0.00        18
Scottish National Party       0.90      0.22      0.35        41

               accuracy                           0.70       528
              macro avg       0.59      0.39      0.42       528
           weighted avg       0.70      0.70      0.66       528

SVM
Macro avg. F1: 0.46722713864306786
                         precision    recall  f1-score   support

           Conservative       0.77      0.90      0.83       312
                 Labour       0.64      0.58      0.61       157
       Liberal Democrat       0.00      0.00      0.00        18
Scottish National Party       0.68      0.32      0.43        41

               accuracy                           0.73       52